In [1]:
import os
import re
import time
import random
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import requests
import pandas as pd
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

In [2]:
BOOKS_BASE_URL = "https://books.toscrape.com/"
PRODUCTS_BASE_URL = "https://web-scraping.dev/products"

BOOKS_MAX_PAGES = 50
PRODUCTS_MAX_PAGES = 6

# Respect the second site's published crawl delay (2 seconds).
BOOKS_DELAY_SECONDS = 1.0
PRODUCTS_DELAY_SECONDS = 2.0

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36 "
        "(Educational web-scraping project)"
    ),
    "Accept-Language": "en-US,en;q=0.9"
}

print("Books website:", BOOKS_BASE_URL)
print("Second website:", PRODUCTS_BASE_URL)

Books website: https://books.toscrape.com/
Second website: https://web-scraping.dev/products


In [3]:
session = requests.Session()
session.headers.update(HEADERS)

retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    status=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

def get_soup(url, session=session):
    """Download one page and return a BeautifulSoup object."""
    response = session.get(url, timeout=25)
    response.raise_for_status()
    response.encoding = response.apparent_encoding
    return BeautifulSoup(response.text, "html.parser"), response

# Test the first website before scraping.
test_soup, test_response = get_soup(BOOKS_BASE_URL)
print("HTTP Status Code:", test_response.status_code)
print("Page title:", test_soup.title.get_text(strip=True) if test_soup.title else "No title")

HTTP Status Code: 200
Page title: All products | Books to Scrape - Sandbox


In [4]:
def check_robots(base_url, target_url, user_agent=HEADERS["User-Agent"]):
    robots_url = urljoin(base_url, "/robots.txt")
    response = session.get(robots_url, timeout=20)

    # A 404 means no robots.txt file is published. Other errors are not treated as permission.
    if response.status_code == 404:
        print(f"No robots.txt found for {base_url}; continuing with low request volume.")
        return None

    response.raise_for_status()

    parser = RobotFileParser()
    parser.set_url(robots_url)
    parser.parse(response.text.splitlines())

    if not parser.can_fetch(user_agent, target_url):
        raise PermissionError(f"robots.txt disallows scraping: {target_url}")

    print(f"robots.txt allows the target URL: {target_url}")
    return parser

books_robots = check_robots(BOOKS_BASE_URL, BOOKS_BASE_URL)
products_robots = check_robots(PRODUCTS_BASE_URL, PRODUCTS_BASE_URL)

No robots.txt found for https://books.toscrape.com/; continuing with low request volume.
robots.txt allows the target URL: https://web-scraping.dev/products


In [5]:
first_page_books, _ = get_soup(BOOKS_BASE_URL)
book_cards = first_page_books.find_all("article", class_="product_pod")

print(f"Found {len(book_cards)} book cards on the first page.")
if book_cards:
    print("First book card:")
    print(book_cards[0].get_text(" ", strip=True))

Found 20 book cards on the first page.
First book card:
A Light in the ... £51.77 In stock Add to basket


In [6]:
RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

def scrape_books_to_scrape(max_pages=BOOKS_MAX_PAGES):
    records = []
    page_url = BOOKS_BASE_URL

    for page_number in range(1, max_pages + 1):
        print(f"Scraping Books to Scrape page {page_number} ...")
        soup, _ = get_soup(page_url)

        cards = soup.find_all("article", class_="product_pod")
        if not cards:
            print("No product cards found. Stopping.")
            break

        for card in cards:
            title_tag = card.select_one("h3 a")
            price_tag = card.select_one(".price_color")
            availability_tag = card.select_one(".availability")
            rating_tag = card.select_one("p.star-rating")

            if title_tag is None or price_tag is None:
                continue

            rating_word = None
            if rating_tag:
                rating_classes = rating_tag.get("class", [])
                rating_word = next(
                    (item for item in rating_classes if item in RATING_MAP),
                    None
                )

            product_url = urljoin(page_url, title_tag.get("href", ""))

            records.append({
                "Product_Name": title_tag.get("title") or title_tag.get_text(" ", strip=True),
                "Price_Raw": price_tag.get_text(" ", strip=True),
                "Price": None,  # Converted during preprocessing
                "Currency": "GBP",
                "Rating_Raw": rating_word,
                "Rating": RATING_MAP.get(rating_word),
                "Availability": availability_tag.get_text(" ", strip=True) if availability_tag else None,
                "Category": "Books",
                "Product_URL": product_url,
                "Source": "Books to Scrape"
            })

        next_link = soup.select_one("li.next a")
        if next_link is None:
            break

        page_url = urljoin(page_url, next_link.get("href"))
        time.sleep(BOOKS_DELAY_SECONDS + random.uniform(0, 0.25))

    return pd.DataFrame(records)

books_df = scrape_books_to_scrape()
print("Books dataset shape:", books_df.shape)
books_df.head()

Scraping Books to Scrape page 1 ...
Scraping Books to Scrape page 2 ...
Scraping Books to Scrape page 3 ...
Scraping Books to Scrape page 4 ...
Scraping Books to Scrape page 5 ...
Scraping Books to Scrape page 6 ...
Scraping Books to Scrape page 7 ...
Scraping Books to Scrape page 8 ...
Scraping Books to Scrape page 9 ...
Scraping Books to Scrape page 10 ...
Scraping Books to Scrape page 11 ...
Scraping Books to Scrape page 12 ...
Scraping Books to Scrape page 13 ...
Scraping Books to Scrape page 14 ...
Scraping Books to Scrape page 15 ...
Scraping Books to Scrape page 16 ...
Scraping Books to Scrape page 17 ...
Scraping Books to Scrape page 18 ...
Scraping Books to Scrape page 19 ...
Scraping Books to Scrape page 20 ...
Scraping Books to Scrape page 21 ...
Scraping Books to Scrape page 22 ...
Scraping Books to Scrape page 23 ...
Scraping Books to Scrape page 24 ...
Scraping Books to Scrape page 25 ...
Scraping Books to Scrape page 26 ...
Scraping Books to Scrape page 27 ...
Scraping B

,Product_Name,Price_Raw,Price,Currency,Rating_Raw,Rating,Availability,Category,Product_URL,Source
0,A Light in the Attic,£51.77,None,GBP,Three,3,In stock,Books,https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html,Books to Scrape
1,Tipping the Velvet,£53.74,None,GBP,One,1,In stock,Books,https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html,Books to Scrape
2,Soumission,£50.10,None,GBP,One,1,In stock,Books,https://books.toscrape.com/catalogue/soumission_998/index.html,Books to Scrape
3,Sharp Objects,£47.82,None,GBP,Four,4,In stock,Books,https://books.toscrape.com/catalogue/sharp-objects_997/index.html,Books to Scrape
4,Sapiens: A Brief History of Humankind,£54.23,None,GBP,Five,5,In stock,Books,https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html,Books to Scrape


In [7]:
second_page_url = f"{PRODUCTS_BASE_URL}?page=1"
second_soup, second_response = get_soup(second_page_url)

print("HTTP Status Code:", second_response.status_code)
print("Page title:", second_soup.title.get_text(strip=True) if second_soup.title else "No title")
print("Page text preview:")
print(second_soup.get_text(" ", strip=True)[:700])

HTTP Status Code: 200
Page title: web-scraping.dev product page 1
Page text preview:
web-scraping.dev product page 1 web-scraping.dev Docs API Graphql Products Reviews Testimonials File Download MCP Tools login Clear Continue all apparel consumables household descending ascending Box of Chocolate Candy Indulge your sweet tooth with our Box of Chocolate Candy. Each box contains an assortment of rich, flavorful chocolates with a smooth, creamy filling. Choose from a variety of flavors including zesty orange and sweet cherry. Whether you're looking for the perfect gift or just want to treat yourself, our Box of Chocolate Candy is sure to satisfy. 24.99 Dark Red Energy Potion Unleash the power within with our 'Dark Red Potion', an energy drink as intense as the games you play. I


In [8]:
PRICE_PATTERN = re.compile(r"(?<![A-Za-z])(?:[$£€]\s*)?\d[\d,]*(?:\.\d{1,2})?")

def extract_price_from_container(container):
    """Return the visible price text from a product container, or None."""
    # Prefer elements whose class or item property identifies a price.
    price_element = container.select_one(
        ".price, [class*='price'], [data-testid*='price'], [itemprop='price']"
    )
    if price_element:
        match = PRICE_PATTERN.search(price_element.get_text(" ", strip=True))
        if match:
            return match.group(0).strip()

    # Fallback: find a numeric price in the container's visible text.
    text = container.get_text(" ", strip=True)
    matches = PRICE_PATTERN.findall(text)
    return matches[-1].strip() if matches else None


def scrape_web_scraping_dev(max_pages=PRODUCTS_MAX_PAGES):
    records = []
    seen_urls = set()

    for page_number in range(1, max_pages + 1):
        page_url = f"{PRODUCTS_BASE_URL}?page={page_number}"
        print(f"Scraping Web Scraping Dev page {page_number} ...")
        soup, _ = get_soup(page_url)

        # Product names are exposed as links inside heading elements.
        headings = soup.find_all(["h2", "h3", "h4"])
        page_records_before = len(records)

        for heading in headings:
            title = heading.get_text(" ", strip=True)
            if not title:
                continue

            link = heading.find("a", href=True)
            product_url = urljoin(page_url, link["href"]) if link else None

            # Avoid navigation headings and duplicate product links.
            if product_url and product_url in seen_urls:
                continue

            container = heading
            price_raw = None
            for _ in range(6):
                price_raw = extract_price_from_container(container)
                if price_raw:
                    break
                if container.parent is None:
                    break
                container = container.parent

            if not price_raw:
                continue

            if product_url:
                seen_urls.add(product_url)

            records.append({
                "Product_Name": title,
                "Price_Raw": price_raw,
                "Price": None,
                # The listing displays a numeric price without a clear currency symbol.
                "Currency": "Not specified",
                "Rating_Raw": None,
                "Rating": None,
                "Availability": None,
                "Category": "General Products",
                "Product_URL": product_url,
                "Source": "Web Scraping Dev"
            })

        if len(records) == page_records_before:
            print("No product records extracted from this page; check the page structure.")
            # Do not silently claim a successful scrape.
            if page_number == 1:
                raise ValueError(
                    "No products were extracted from Web Scraping Dev. "
                    "Inspect the page HTML and update the selectors."
                )
            break

        if page_number < max_pages:
            time.sleep(PRODUCTS_DELAY_SECONDS)

    return pd.DataFrame(records)

products_df = scrape_web_scraping_dev()
print("Web Scraping Dev dataset shape:", products_df.shape)
products_df.head()

Scraping Web Scraping Dev page 1 ...
Scraping Web Scraping Dev page 2 ...
Scraping Web Scraping Dev page 3 ...
Scraping Web Scraping Dev page 4 ...
Scraping Web Scraping Dev page 5 ...
Scraping Web Scraping Dev page 6 ...
No product records extracted from this page; check the page structure.
Web Scraping Dev dataset shape: (25, 10)


,Product_Name,Price_Raw,Price,Currency,Rating_Raw,Rating,Availability,Category,Product_URL,Source
0,Box of Chocolate Candy,24.99,None,Not specified,None,None,None,General Products,https://web-scraping.dev/product/1,Web Scraping Dev
1,Dark Red Energy Potion,4.99,None,Not specified,None,None,None,General Products,https://web-scraping.dev/product/2,Web Scraping Dev
2,Teal Energy Potion,4.99,None,Not specified,None,None,None,General Products,https://web-scraping.dev/product/3,Web Scraping Dev
3,Red Energy Potion,4.99,None,Not specified,None,None,None,General Products,https://web-scraping.dev/product/4,Web Scraping Dev
4,Blue Energy Potion,4.99,None,Not specified,None,None,None,General Products,https://web-scraping.dev/product/5,Web Scraping Dev


In [9]:
print("Books to Scrape:")
print("Shape:", books_df.shape)
display(books_df.head())

print("\nWeb Scraping Dev:")
print("Shape:", products_df.shape)
display(products_df.head())

assert "Source" in books_df.columns and "Source" in products_df.columns

Books to Scrape:
Shape: (1000, 10)


,Product_Name,Price_Raw,Price,Currency,Rating_Raw,Rating,Availability,Category,Product_URL,Source
0,A Light in the Attic,£51.77,None,GBP,Three,3,In stock,Books,https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html,Books to Scrape
1,Tipping the Velvet,£53.74,None,GBP,One,1,In stock,Books,https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html,Books to Scrape
2,Soumission,£50.10,None,GBP,One,1,In stock,Books,https://books.toscrape.com/catalogue/soumission_998/index.html,Books to Scrape
3,Sharp Objects,£47.82,None,GBP,Four,4,In stock,Books,https://books.toscrape.com/catalogue/sharp-objects_997/index.html,Books to Scrape
4,Sapiens: A Brief History of Humankind,£54.23,None,GBP,Five,5,In stock,Books,https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html,Books to Scrape



Web Scraping Dev:
Shape: (25, 10)


,Product_Name,Price_Raw,Price,Currency,Rating_Raw,Rating,Availability,Category,Product_URL,Source
0,Box of Chocolate Candy,24.99,None,Not specified,None,None,None,General Products,https://web-scraping.dev/product/1,Web Scraping Dev
1,Dark Red Energy Potion,4.99,None,Not specified,None,None,None,General Products,https://web-scraping.dev/product/2,Web Scraping Dev
2,Teal Energy Potion,4.99,None,Not specified,None,None,None,General Products,https://web-scraping.dev/product/3,Web Scraping Dev
3,Red Energy Potion,4.99,None,Not specified,None,None,None,General Products,https://web-scraping.dev/product/4,Web Scraping Dev
4,Blue Energy Potion,4.99,None,Not specified,None,None,None,General Products,https://web-scraping.dev/product/5,Web Scraping Dev


In [11]:
STANDARD_COLUMNS = [
    "Product_Name",
    "Price_Raw",
    "Price",
    "Currency",
    "Rating_Raw",
    "Rating",
    "Availability",
    "Category",
    "Product_URL",
    "Source"
]

books_df = books_df.reindex(columns=STANDARD_COLUMNS)
products_df = products_df.reindex(columns=STANDARD_COLUMNS)

combined_raw_df = pd.concat(
    [books_df, products_df],
    ignore_index=True
)

print("Combined dataset shape:", combined_raw_df.shape)
display(combined_raw_df.head(100))
print("\nRecords by source:")
print(combined_raw_df["Source"].value_counts(dropna=False))

Combined dataset shape: (1025, 10)


,Product_Name,Price_Raw,Price,Currency,Rating_Raw,Rating,Availability,Category,Product_URL,Source
0,A Light in the Attic,£51.77,None,GBP,Three,3,In stock,Books,https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html,Books to Scrape
1,Tipping the Velvet,£53.74,None,GBP,One,1,In stock,Books,https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html,Books to Scrape
2,Soumission,£50.10,None,GBP,One,1,In stock,Books,https://books.toscrape.com/catalogue/soumission_998/index.html,Books to Scrape
3,Sharp Objects,£47.82,None,GBP,Four,4,In stock,Books,https://books.toscrape.com/catalogue/sharp-objects_997/index.html,Books to Scrape
4,Sapiens: A Brief History of Humankind,£54.23,None,GBP,Five,5,In stock,Books,https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html,Books to Scrape
...,...,...,...,...,...,...,...,...,...,...
95,Lumberjanes Vol. 3: A Terrible Plan (Lumberjanes #9-12),£19.92,None,GBP,Two,2,In stock,Books,https://books.toscrape.com/catalogue/lumberjanes-vol-3-a-terrible-plan-lumberjanes-9-12_905/inde...,Books to Scrape
96,"Layered: Baking, Building, and Styling Spectacular Cakes",£40.11,None,GBP,One,1,In stock,Books,https://books.toscrape.com/catalogue/layered-baking-building-and-styling-spectacular-cakes_904/i...,Books to Scrape
97,Judo: Seven Steps to Black Belt (an Introductory Guide for Beginners),£53.90,None,GBP,Two,2,In stock,Books,https://books.toscrape.com/catalogue/judo-seven-steps-to-black-belt-an-introductory-guide-for-be...,Books to Scrape
98,Join,£35.67,None,GBP,Five,5,In stock,Books,https://books.toscrape.com/catalogue/join_902/index.html,Books to Scrape



Records by source:
Source
Books to Scrape     1000
Web Scraping Dev      25
Name: count, dtype: int64


In [12]:
print("BEFORE PREPROCESSING — HEAD")
display(combined_raw_df.head())

print("\nBEFORE PREPROCESSING — INFO")
combined_raw_df.info()

print("\nBEFORE PREPROCESSING — DESCRIBE")
display(combined_raw_df.describe(include="all").T)

print("\nBEFORE PREPROCESSING — MISSING VALUES")
display(combined_raw_df.isnull().sum().to_frame("Missing_Count"))

BEFORE PREPROCESSING — HEAD


,Product_Name,Price_Raw,Price,Currency,Rating_Raw,Rating,Availability,Category,Product_URL,Source
0,A Light in the Attic,£51.77,None,GBP,Three,3,In stock,Books,https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html,Books to Scrape
1,Tipping the Velvet,£53.74,None,GBP,One,1,In stock,Books,https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html,Books to Scrape
2,Soumission,£50.10,None,GBP,One,1,In stock,Books,https://books.toscrape.com/catalogue/soumission_998/index.html,Books to Scrape
3,Sharp Objects,£47.82,None,GBP,Four,4,In stock,Books,https://books.toscrape.com/catalogue/sharp-objects_997/index.html,Books to Scrape
4,Sapiens: A Brief History of Humankind,£54.23,None,GBP,Five,5,In stock,Books,https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html,Books to Scrape



BEFORE PREPROCESSING — INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Product_Name  1025 non-null   object
 1   Price_Raw     1025 non-null   object
 2   Price         0 non-null      object
 3   Currency      1025 non-null   object
 4   Rating_Raw    1000 non-null   object
 5   Rating        1000 non-null   object
 6   Availability  1000 non-null   object
 7   Category      1025 non-null   object
 8   Product_URL   1025 non-null   object
 9   Source        1025 non-null   object
dtypes: object(10)
memory usage: 80.2+ KB

BEFORE PREPROCESSING — DESCRIBE


,count,unique,top,freq
Product_Name,1025,1011,Box of Chocolate Candy,3
Price_Raw,1025,911,4.99,10
Price,0,0,NaN,NaN
Currency,1025,2,GBP,1000
Rating_Raw,1000,5,One,226
Rating,1000,5,1,226
Availability,1000,1,In stock,1000
Category,1025,2,Books,1000
Product_URL,1025,1025,https://web-scraping.dev/product/25,1
Source,1025,2,Books to Scrape,1000



BEFORE PREPROCESSING — MISSING VALUES


,Missing_Count
Product_Name,0
Price_Raw,0
Price,1025
Currency,0
Rating_Raw,25
Rating,25
Availability,25
Category,0
Product_URL,0
Source,0


In [16]:
clean_df = combined_raw_df.copy()

def clean_text(value):
    if pd.isna(value):
        return pd.NA
    value = re.sub(r"\s+", " ", str(value)).strip()
    return value if value else pd.NA

text_columns = [
    "Product_Name", "Price_Raw", "Currency",
    "Rating_Raw", "Availability", "Category", "Source"
]

for column in text_columns:
    clean_df[column] = clean_df[column].apply(clean_text)

# Standardize categorical text while preserving readable product names.
clean_df["Category"] = clean_df["Category"].str.strip().str.title()
clean_df["Source"] = clean_df["Source"].str.strip()
clean_df["Availability"] = clean_df["Availability"].str.lower()

display(clean_df.head())

,Product_Name,Price_Raw,Price,Currency,Rating_Raw,Rating,Availability,Category,Product_URL,Source
0,A Light in the Attic,£51.77,None,GBP,Three,3,in stock,Books,https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html,Books to Scrape
1,Tipping the Velvet,£53.74,None,GBP,One,1,in stock,Books,https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html,Books to Scrape
2,Soumission,£50.10,None,GBP,One,1,in stock,Books,https://books.toscrape.com/catalogue/soumission_998/index.html,Books to Scrape
3,Sharp Objects,£47.82,None,GBP,Four,4,in stock,Books,https://books.toscrape.com/catalogue/sharp-objects_997/index.html,Books to Scrape
4,Sapiens: A Brief History of Humankind,£54.23,None,GBP,Five,5,in stock,Books,https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html,Books to Scrape


In [17]:
def parse_numeric_price(value):
    if pd.isna(value):
        return pd.NA
    # Remove currency symbols and thousands separators; retain decimal point.
    cleaned = re.sub(r"[^0-9.]", "", str(value).replace(",", ""))
    try:
        return float(cleaned) if cleaned else pd.NA
    except ValueError:
        return pd.NA

clean_df["Price"] = clean_df["Price_Raw"].apply(parse_numeric_price)
clean_df["Price"] = pd.to_numeric(clean_df["Price"], errors="coerce")
clean_df["Rating"] = pd.to_numeric(clean_df["Rating"], errors="coerce")

# Explicitly normalize stock status where the source provides it.
clean_df["Availability_Status"] = clean_df["Availability"].apply(
    lambda value: (
        "in_stock" if isinstance(value, str) and "in stock" in value
        else "out_of_stock" if isinstance(value, str) and "out of stock" in value
        else "not_provided" if pd.isna(value)
        else "other"
    )
)

print(clean_df[["Price_Raw", "Price", "Currency", "Rating_Raw", "Rating"]].head(10))
print("\nData types:")
print(clean_df.dtypes)

  Price_Raw  Price Currency Rating_Raw  Rating
0    £51.77  51.77      GBP      Three     3.0
1    £53.74  53.74      GBP        One     1.0
2    £50.10  50.10      GBP        One     1.0
3    £47.82  47.82      GBP       Four     4.0
4    £54.23  54.23      GBP       Five     5.0
5    £22.65  22.65      GBP        One     1.0
6    £33.34  33.34      GBP       Four     4.0
7    £17.93  17.93      GBP      Three     3.0
8    £22.60  22.60      GBP       Four     4.0
9    £52.15  52.15      GBP        One     1.0

Data types:
Product_Name            object
Price_Raw               object
Price                  float64
Currency                object
Rating_Raw              object
Rating                 float64
Availability            object
Category                object
Product_URL             object
Source                  object
Availability_Status     object
dtype: object


In [18]:
print("Missing values before handling:")
display(clean_df.isnull().sum().to_frame("Missing_Before"))

rows_before = len(clean_df)

clean_df = clean_df.dropna(subset=["Product_Name", "Price"])
clean_df = clean_df[clean_df["Price"] >= 0].copy()

# Make missing text fields explicit for readability; numeric rating remains NaN.
clean_df["Rating_Raw"] = clean_df["Rating_Raw"].fillna("Not provided")
clean_df["Availability"] = clean_df["Availability"].fillna("Not provided")

print(f"Rows before required-field filtering: {rows_before}")
print(f"Rows after required-field filtering: {len(clean_df)}")
print("\nMissing values after handling:")
display(clean_df.isnull().sum().to_frame("Missing_After"))

Missing values before handling:


,Missing_Before
Product_Name,0
Price_Raw,0
Price,0
Currency,0
Rating_Raw,25
Rating,25
Availability,25
Category,0
Product_URL,0
Source,0


Rows before required-field filtering: 1025
Rows after required-field filtering: 1025

Missing values after handling:


,Missing_After
Product_Name,0
Price_Raw,0
Price,0
Currency,0
Rating_Raw,0
Rating,25
Availability,0
Category,0
Product_URL,0
Source,0


In [19]:
rows_before_dedup = len(clean_df)

clean_df = clean_df.drop_duplicates()

clean_df = clean_df.drop_duplicates(
    subset=["Source", "Product_URL"],
    keep="first"
)

print("Rows before deduplication:", rows_before_dedup)
print("Rows after deduplication:", len(clean_df))
print("Duplicates removed:", rows_before_dedup - len(clean_df))

Rows before deduplication: 1025
Rows after deduplication: 1025
Duplicates removed: 0


In [20]:
print("AFTER PREPROCESSING — HEAD")
display(clean_df.head())

print("\nAFTER PREPROCESSING — INFO")
clean_df.info()

print("\nAFTER PREPROCESSING — DESCRIBE")
display(clean_df.describe(include="all").T)

print("\nAFTER PREPROCESSING — MISSING VALUES")
display(clean_df.isnull().sum().to_frame("Missing_Count"))

print("\nFinal dataset shape:", clean_df.shape)

AFTER PREPROCESSING — HEAD


,Product_Name,Price_Raw,Price,Currency,Rating_Raw,Rating,Availability,Category,Product_URL,Source,Availability_Status
0,A Light in the Attic,£51.77,51.77,GBP,Three,3.0,in stock,Books,https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html,Books to Scrape,in_stock
1,Tipping the Velvet,£53.74,53.74,GBP,One,1.0,in stock,Books,https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html,Books to Scrape,in_stock
2,Soumission,£50.10,50.10,GBP,One,1.0,in stock,Books,https://books.toscrape.com/catalogue/soumission_998/index.html,Books to Scrape,in_stock
3,Sharp Objects,£47.82,47.82,GBP,Four,4.0,in stock,Books,https://books.toscrape.com/catalogue/sharp-objects_997/index.html,Books to Scrape,in_stock
4,Sapiens: A Brief History of Humankind,£54.23,54.23,GBP,Five,5.0,in stock,Books,https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html,Books to Scrape,in_stock



AFTER PREPROCESSING — INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Product_Name         1025 non-null   object 
 1   Price_Raw            1025 non-null   object 
 2   Price                1025 non-null   float64
 3   Currency             1025 non-null   object 
 4   Rating_Raw           1025 non-null   object 
 5   Rating               1000 non-null   float64
 6   Availability         1025 non-null   object 
 7   Category             1025 non-null   object 
 8   Product_URL          1025 non-null   object 
 9   Source               1025 non-null   object 
 10  Availability_Status  1025 non-null   object 
dtypes: float64(2), object(9)
memory usage: 88.2+ KB

AFTER PREPROCESSING — DESCRIBE


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Product_Name,1025,1011,Box of Chocolate Candy,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Price_Raw,1025,911,4.99,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Price,1025.0,NaN,NaN,NaN,34.970829,15.001765,4.99,21.96,35.67,47.61,89.99
Currency,1025,2,GBP,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Rating_Raw,1025,6,One,226,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Rating,1000.0,NaN,NaN,NaN,2.923,1.434967,1.0,2.0,3.0,4.0,5.0
Availability,1025,2,in stock,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,1025,2,Books,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product_URL,1025,1025,https://web-scraping.dev/product/25,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Source,1025,2,Books to Scrape,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN



AFTER PREPROCESSING — MISSING VALUES


,Missing_Count
Product_Name,0
Price_Raw,0
Price,0
Currency,0
Rating_Raw,0
Rating,25
Availability,0
Category,0
Product_URL,0
Source,0



Final dataset shape: (1025, 11)


In [21]:
print("Number of products by source:")
display(clean_df["Source"].value_counts().to_frame("Product_Count"))

print("\nAverage price by source and currency:")
price_summary = (
    clean_df.groupby(["Source", "Currency"])["Price"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)
display(price_summary)

print("\nRating distribution (where available):")
display(clean_df["Rating"].value_counts(dropna=False).sort_index())

print("\nAvailability status:")
display(clean_df["Availability_Status"].value_counts())

Number of products by source:


,Product_Count
Source,
Books to Scrape,1000
Web Scraping Dev,25



Average price by source and currency:


,,count,mean,median,min,max
Source,Currency,,,,,
Books to Scrape,GBP,1000,35.07,35.98,10.00,59.99
Web Scraping Dev,Not specified,25,30.99,24.99,4.99,89.99



Rating distribution (where available):


Rating
1.0    226
2.0    196
3.0    203
4.0    179
5.0    196
NaN     25
Name: count, dtype: int64


Availability status:


Availability_Status
in_stock        1000
not_provided      25
Name: count, dtype: int64

In [22]:
os.makedirs("data", exist_ok=True)

combined_raw_df.to_csv(
    "data/combined_products_before_preprocessing.csv",
    index=False,
    encoding="utf-8-sig"
)

clean_df.to_csv(
    "data/combined_products_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Files saved successfully:")
print("- data/combined_products_before_preprocessing.csv")
print("- data/combined_products_cleaned.csv")

Files saved successfully:
- data/combined_products_before_preprocessing.csv
- data/combined_products_cleaned.csv
